02-Ground Truth

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [39]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [40]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [5]:
documents = documents_llm

In [42]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]
#"Questions" is used to set the attribute "text_format", in our call to OpenAI's "responses" API.

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
#Prepare the document as JSON
import json
user_prompt = json.dumps(doc)

In [11]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
response.output_parsed.questions

['I just found this course late — can I still enroll and catch up, or is it too late to join?',
 'If I start the course after it has already been running, am I still able to participate normally?',
 'Am I allowed to join the course now even though it already started?',
 'What do I need to do if I want to get the certificate after joining the course late?',
 'Is it still possible to receive the certificate if I only discovered the course after it began?']

In [15]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [16]:
from evaluation_utils import llm_structured

In [17]:
#This is just wrapping the above into a function:
#Asking the llm to generate 5 questions: "data_gen_instructions" is the prompt, 
#"user_prompt" is the FAQ record, 
# and "Questions" is the output format.
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions[0])
print(result.questions[1])
print(result.questions[2])
print(result.questions[3])
print(result.questions[4])

I just found this course, is it too late for me to start or can I still join now?
Can new students still enroll in the course after it has already started?
If I join the course late, will I still be able to get a certificate?
What do I need to do to make sure I can still receive the course certificate if I start now?
Is there a deadline for project submission if I want to be eligible for the certificate?


In [18]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=106, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=313)

In [19]:
from evaluation_utils import calc_price

In [20]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000477, 'total_cost': 0.00063225}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"] # doc is the particular FAQ record, that we used it's "answer" to get llm to generate 5 questions.
    })                        # now we just attached the id

records

[{'question': 'I just found this course, is it too late for me to start or can I still join now?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still enroll in the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to make sure I can still receive the course certificate if I start now?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for project submission if I want to be eligible for the certificate?',
  'document': '74eb249bbf'}]

03-Ground Truth Batch

In [22]:
import pandas as pd

In [23]:
pd.DataFrame(records) 
#records is the list of 5 questions generated by the llm, 
#and the id of the FAQ record that was used to generate them.

,question,document
0,"I just found this course, is it too late for m...",74eb249bbf
1,Can new students still enroll in the course af...,74eb249bbf
2,"If I join the course late, will I still be abl...",74eb249bbf
3,What do I need to do to make sure I can still ...,74eb249bbf
4,Is there a deadline for project submission if ...,74eb249bbf


In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
#A.Create a function that takes a FAQ record, 
#then call "llm_structured_retry" to generate 5 questions based on the FAQ record.

#This will then be used in a loop, where each and evry "doc" in documents will be passed to this function,
# and 5 questions will be generated for each FAQ record.

#B.Then it stores the generated questions against the FAQ record id, 
#in a list of dictionaries called "records".

#So, this function wrapps up A and B, which was demonstrated in the previous module.

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I found it late?',
   'document': '74eb249bbf'},
  {'question': 'Is it too late to join llm-zoomcamp now?',
   'document': '74eb249bbf'},
  {'question': 'If I start the course after it began, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to submit my project before submissions close to get the certificate?',
   'document': '74eb249bbf'},
  {'question': 'I just found this course — can I still participate, and what do I need for the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=299))

In [44]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]): # ":5" = slice up to index 5, but not including index 5, so it will generate questions for the first 5 FAQ records.
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [45]:
len(usages)

5

In [28]:
#Parallel Processing!
#Because doing 5 documents took 10 seconds, doing all of the 1xx documents will be crazy slow!
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [29]:
#max_workers=6 means that 6 threads will be used to process the documents in parallel.

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)
    #map_progress is a helper function in "evaluation_utils.py"
    # that shows a progress bar while processing the documents in parallel.

#After this, all docuemnts have 5 dummy questions generated by the llm, and stored in "results".


  0%|          | 0/113 [00:00<?, ?it/s]

In [30]:
len(results)
#results[1]

113

In [31]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records) #extend = every single element within the list
    usages.append(usage) #append = one element at a time

len(ground_truth)

565

In [32]:
ground_truth[10]

{'question': 'How do I join the Office Hours or workshop stream if I’m a student and don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [33]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.088359

In [34]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.088359

In [35]:
df_ground_truth = pd.DataFrame(ground_truth)

In [36]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [37]:
len(df_ground_truth)

565

In [38]:
import os

# 1. Check if Python thinks the file exists
file_path = r"C:\data\ground_truth.csv"
print(r"Does Python see the file at C:\data\ground_truth.csv?:", os.path.exists(file_path))

# 2. Check your current working directory
print("Where your notebook is actually looking:", os.getcwd())


Does Python see the file at C:\data\ground_truth.csv?: False
Where your notebook is actually looking: /workspaces/llm-zoomcamp-2026-code/Module 4 Evaluation
